# Decision Tree Classifier — Telecom Customer Churn

**Beginner-to-deployment learning notebook**  
Project 1 of 2 for the Decision Tree algorithm.

> **Learning goal:** understand the theory and mathematics, inspect a documented dataset, train and evaluate a classifier, track experiments, save a reproducible pipeline, test it, and prepare it for a Streamlit app. Deployment will be a later checkpoint after the model and artifacts pass validation.

## How to use this notebook

1. Read each explanation before running its code cell.
2. Run cells from top to bottom.
3. Inspect every output; do not skip errors.
4. Record any changed assumptions or results in the project README.
5. At the end, save the notebook with outputs, then export HTML/PDF using `tools/export_notebook.py`.

## Learning lifecycle covered here

1. Problem definition and intended use.
2. Data source, licence/terms, and data dictionary.
3. Data inspection and exploratory analysis.
4. Data quality and preprocessing.
5. Train/test split and leakage prevention.
6. Decision Tree theory and baseline comparison.
7. Training and hyperparameter selection.
8. Evaluation and error analysis.
9. MLflow experiment tracking.
10. Pipeline and artifact saving.
11. Tests and app-readiness checks.
12. Streamlit deployment (a later step, after local validation).


## 1. Problem definition — what, why, and when?

**Classification** predicts a discrete class such as `Churn = Yes` or `Churn = No`. In this project, the target is the historical churn label in a telecom sample dataset. The model learns patterns associated with that label; it does not prove why a customer left and cannot guarantee future behaviour.

**Why a Decision Tree?** A tree repeatedly asks feature-based questions and routes each row down branches to a predicted class. It can model nonlinear relationships and feature interactions, needs little feature scaling, and can be explained through paths such as “if contract type is month-to-month and tenure is short, the predicted class may be churn.” Actual rules depend on the fitted data.

**When it is useful:** tabular classification, nonlinear feature relationships, interpretable rule paths, and as a baseline for tree ensembles.

**When to be careful:** small data, noisy features, class imbalance, unstable splits, high-cardinality identifiers, data leakage, and trees grown too deep. A single tree can overfit; use validation and pruning/depth constraints.

**Problem statement:** build a reproducible educational classifier that predicts the dataset's churn label from features available at prediction time. Exclude customer identifiers and any information only known after churn. Do not use it as the sole basis for consequential customer decisions.

## 2. Mathematics: how a Decision Tree chooses a split

At a node, the algorithm considers candidate splits and scores how pure the child nodes become. A pure node contains one class; a mixed node contains multiple classes.

### Gini impurity

For a node with class proportions \(p_1, p_2, \ldots, p_K\):

\[
Gini(t) = 1 - \sum_{k=1}^{K} p_k^2
\]

For binary classes with proportions \(p\) and \(1-p\), this becomes \(2p(1-p)\). It is zero when all observations belong to one class and is largest when classes are evenly mixed.

### Entropy

\[
H(t) = -\sum_{k=1}^{K} p_k \log_2(p_k)
\]

Use the convention \(0\log_2(0)=0\). Entropy is also zero for a pure node and higher for a mixed node.

### Weighted impurity after a split

If a parent node \(t\) is split into children \(t_1,\ldots,t_m\):

\[
I_{split}(t) = \sum_{j=1}^{m}\frac{N_j}{N} I(t_j)
\]

where \(N\) is the number of rows in the parent, \(N_j\) is the number in child \(j\), and \(I\) is Gini or entropy.

### Impurity decrease

\[
\Delta I = I(t) - I_{split}(t)
\]

The tree searches for splits that reduce weighted impurity. Scikit-learn supports `criterion="gini"` and `criterion="entropy"` (also `log_loss` in supported versions). We will compare choices using validation results rather than assuming one is always best.

### Prediction at a leaf

The predicted class is normally the most frequent training class in the leaf. `predict_proba` reports class probabilities based on the class distribution in the leaf; these probabilities are not automatically well calibrated.

### Complexity and overfitting controls

- `max_depth`: maximum depth of the tree.
- `min_samples_split`: minimum samples required to split a node.
- `min_samples_leaf`: minimum samples allowed in a leaf.
- `max_leaf_nodes`: maximum number of leaves.
- `ccp_alpha`: cost-complexity pruning strength; larger values prune more.

Deeper trees can capture more detail but may memorize noise. Select complexity with training-only/validation workflows or cross-validation, and preserve a final test set for the final evaluation.

In [ ]:
# A tiny hand-calculated example: Gini impurity for a binary node.
# Suppose a node contains 6 churn cases and 4 non-churn cases.
p_churn = 6 / 10
p_no_churn = 4 / 10
gini = 1 - (p_churn ** 2 + p_no_churn ** 2)
print(f"Churn proportion: {p_churn:.2f}")
print(f"No-churn proportion: {p_no_churn:.2f}")
print(f"Gini impurity: {gini:.3f}")

# Check: impurity is between 0 and 0.5 for a binary classification node.
assert 0 <= gini <= 0.5


## 3. Dataset source and licence checkpoint

Candidate dataset: IBM's sample Telco Customer Churn CSV. The target is commonly named `Churn` and has values such as `Yes` and `No`. The source copy is commonly described as containing 7,043 customer records, but this notebook will verify the downloaded file rather than assuming those numbers.

- Dataset context: [IBM telecom churn tutorial](https://www.ibm.com/docs/en/ws-and-kc?topic=tutorials-predict-telecommunications-churn)
- Candidate CSV source: [Telco-Customer-Churn.csv](https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv)

**Licence caution:** a repository's code licence does not necessarily license a dataset stored in that repository. Review the dataset's applicable terms before redistribution. This notebook downloads the data for local educational analysis and does not automatically commit the raw CSV to GitHub.

Run the next cell to download a local copy. If the URL becomes unavailable or its terms change, stop and verify the source rather than silently switching datasets.

In [ ]:
from pathlib import Path
from urllib.request import urlopen

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / "Telco-Customer-Churn.csv"
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

if not DATA_PATH.exists():
    print("Downloading candidate dataset from the documented source...")
    with urlopen(DATA_URL, timeout=30) as response:
        DATA_PATH.write_bytes(response.read())
else:
    print(f"Using existing local dataset: {DATA_PATH}")

print(f"Dataset file exists: {DATA_PATH.exists()}")
print(f"File size: {DATA_PATH.stat().st_size:,} bytes")


## 4. Data inspection and exploratory analysis

Before modelling, check row/column counts, column names, types, missing values, duplicates, target balance, and numeric summaries. These checks help detect parsing errors and inform preprocessing.

Customer identifiers such as `customerID` should not be used as predictive features. They identify records rather than customer behaviour and can encourage spurious patterns. We will also inspect `TotalCharges`, which may arrive as text because some records contain blanks.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(DATA_PATH)
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print(f"Duplicate rows: {df.duplicated().sum():,}")

if "Churn" not in df.columns:
    raise ValueError("Expected target column 'Churn' was not found. Re-check the source file.")

display(df["Churn"].value_counts(dropna=False).to_frame("count"))
ax = df["Churn"].value_counts(normalize=True).plot(kind="bar", title="Target class proportions")
ax.set_xlabel("Churn")
ax.set_ylabel("Proportion")
plt.tight_layout()
plt.show()


## 5. Data preparation and leakage prevention

We will separate the target from features and exclude identifiers. Convert `TotalCharges` to numeric, allowing unparseable blank strings to become missing values; then handle those missing values inside the preprocessing pipeline.

Categorical features will be one-hot encoded, numeric features will have missing values imputed, and the split will be stratified so the churn proportion is approximately preserved in train and test sets. Tree models generally do not require standard scaling, so we will not add scaling without a reason.

**Leakage rule:** learn imputation values and category encodings from training data only. A scikit-learn `Pipeline`/`ColumnTransformer` keeps those fitted steps inside the model artifact.

In [ ]:
from sklearn.model_selection import train_test_split

data = df.copy()
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
data = data.drop_duplicates().copy()

y = data["Churn"].astype(str).str.strip()
X = data.drop(columns=["Churn", "customerID"], errors="ignore")

# Validate that the target contains the expected two classes.
classes = sorted(y.dropna().unique().tolist())
print("Target classes:", classes)
if set(classes) != {"No", "Yes"}:
    raise ValueError(f"Unexpected target labels: {classes}. Inspect the dataset before continuing.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print("Train churn proportion:", round((y_train == "Yes").mean(), 3))
print("Test churn proportion:", round((y_test == "Yes").mean(), 3))


## 6. Build preprocessing and a simple baseline

A baseline gives us a reference point. The `DummyClassifier` predicts according to a simple strategy and helps answer whether a more complex model adds useful predictive value. We will compare metrics beyond accuracy because churn classes may be imbalanced.

The `ColumnTransformer` applies median imputation to numeric columns and most-frequent imputation plus one-hot encoding to categorical columns. `handle_unknown="ignore"` prevents the pipeline from crashing when the app receives a category not observed during training.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

baseline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent")),
])
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
print(f"Baseline accuracy: {accuracy_score(y_test, baseline_pred):.3f}")
print(f"Baseline churn recall: {recall_score(y_test, baseline_pred, pos_label='Yes', zero_division=0):.3f}")
print(f"Baseline churn F1: {f1_score(y_test, baseline_pred, pos_label='Yes', zero_division=0):.3f}")


## 7. Train a Decision Tree Classifier

We begin with a controlled tree (`max_depth`, `min_samples_leaf`) rather than an unrestricted tree. These hyperparameters regularize the model and can reduce overfitting. We use `class_weight="balanced"` as a candidate for the imbalanced target; it changes training weights and must be evaluated rather than assumed to be beneficial.

**Important:** the test set is held out for final evaluation. Hyperparameter tuning and threshold choices should be made using cross-validation or a validation split inside the training data, not by repeatedly optimising against the test set.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        criterion="gini",
        max_depth=5,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
    )),
])
tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)
print(f"Accuracy:  {accuracy_score(y_test, tree_pred):.3f}")
print(f"Precision: {precision_score(y_test, tree_pred, pos_label='Yes', zero_division=0):.3f}")
print(f"Recall:    {recall_score(y_test, tree_pred, pos_label='Yes', zero_division=0):.3f}")
print(f"F1-score:  {f1_score(y_test, tree_pred, pos_label='Yes', zero_division=0):.3f}")

if "Yes" in tree_model.classes_ if hasattr(tree_model, "classes_") else False:
    churn_index = list(tree_model.classes_).index("Yes")
    churn_scores = tree_model.predict_proba(X_test)[:, churn_index]
    print(f"ROC-AUC:   {roc_auc_score((y_test == 'Yes').astype(int), churn_scores):.3f}")
    print(f"PR-AUC:    {average_precision_score((y_test == 'Yes').astype(int), churn_scores):.3f}")


## 8. Evaluate the classifier properly

- **Accuracy:** fraction of all predictions that are correct; can be misleading under imbalance.
- **Precision:** among records predicted as churn, the fraction that truly churned.
- **Recall:** among actual churn records, the fraction detected by the model.
- **F1-score:** harmonic mean of precision and recall.
- **ROC-AUC:** ranking quality across thresholds.
- **Average precision / PR-AUC summary:** useful when the positive class is relatively rare.

The appropriate trade-off depends on the use case. We should explain the cost of false positives and false negatives instead of choosing a metric solely because it is high.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

print(classification_report(y_test, tree_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, tree_pred, labels=["No", "Yes"], colorbar=False)
plt.title("Decision Tree — test-set confusion matrix")
plt.tight_layout()
plt.show()


## 9. Inspect feature importance and the learned tree

Tree feature importance is based on the reduction in the selected impurity criterion. It can be biased toward certain feature types and does not establish causation. We will label the transformed feature names and treat the chart as a diagnostic, not as proof of causal influence.

The model should be compared with alternatives and validated before feature importance is used to communicate findings.

In [ ]:
fitted_preprocessor = tree_model.named_steps["preprocessor"]
fitted_tree = tree_model.named_steps["classifier"]
feature_names = fitted_preprocessor.get_feature_names_out()
importance = pd.Series(fitted_tree.feature_importances_, index=feature_names).sort_values(ascending=False).head(15)

ax = importance.sort_values().plot(kind="barh", figsize=(9, 6), title="Top Decision Tree feature importances")
ax.set_xlabel("Impurity-based importance")
plt.tight_layout()
plt.show()


## 10. Track the experiment with MLflow

MLflow records the configuration and metrics so experiments can be compared. Start the local tracking UI from the project environment with `mlflow ui` and open the displayed local URL (typically `http://127.0.0.1:5000`). Do not commit the generated `mlruns/` directory unless you deliberately choose that storage strategy; add it to `.gitignore`.

This cell logs the current run. Later, we can add cross-validation and a parameter search and compare those runs.

In [ ]:
# If MLflow is not installed, run in your project terminal: python -m pip install mlflow
import mlflow
import mlflow.sklearn

mlflow.set_experiment("decision-tree-telecom-churn")
with mlflow.start_run(run_name="decision-tree-depth5-leaf20"):
    mlflow.log_params({
        "criterion": "gini",
        "max_depth": 5,
        "min_samples_leaf": 20,
        "class_weight": "balanced",
        "test_size": 0.20,
        "random_state": 42,
    })
    mlflow.log_metrics({
        "accuracy": accuracy_score(y_test, tree_pred),
        "precision_churn": precision_score(y_test, tree_pred, pos_label="Yes", zero_division=0),
        "recall_churn": recall_score(y_test, tree_pred, pos_label="Yes", zero_division=0),
        "f1_churn": f1_score(y_test, tree_pred, pos_label="Yes", zero_division=0),
    })
    mlflow.sklearn.log_model(tree_model, artifact_path="model")
    print("Logged run:", mlflow.active_run().info.run_id)


## 11. Save the full pipeline and metadata

Save the complete fitted pipeline, not only the Decision Tree estimator. The preprocessing steps must be identical at training and inference time. Model artifacts are executable serialized objects: load only files you trust, and keep library versions compatible.

The metadata records the target, positive class, feature schema and the educational nature of the model. The Streamlit app will later load these files from paths relative to `app.py`, never from an assumed working directory.

In [ ]:
import json
import joblib
import sklearn
import sys

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "decision_tree_churn_pipeline.joblib"
METADATA_PATH = ARTIFACT_DIR / "model_metadata.json"

joblib.dump(tree_model, MODEL_PATH)
metadata = {
    "algorithm": "DecisionTreeClassifier",
    "project": "telecom-customer-churn",
    "target_column": "Churn",
    "positive_class": "Yes",
    "input_features": X.columns.tolist(),
    "classes": [str(value) for value in tree_model.classes_],
    "sklearn_version": sklearn.__version__,
    "python_version": sys.version.split()[0],
    "dataset_source": DATA_URL,
    "educational_only": True,
    "limitations": [
        "Historical sample data; not a guarantee of future churn.",
        "Feature importance is not causal evidence.",
        "Evaluate subgroup performance and operational costs before any real deployment."
    ]
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved model:", MODEL_PATH.resolve())
print("Saved metadata:", METADATA_PATH.resolve())


## 12. Validate artifact loading and predictions

Before app development, prove that the saved pipeline can be loaded and predicts from raw feature rows. This check catches missing files, broken serialization and schema mismatches.

In [ ]:
loaded_model = joblib.load(MODEL_PATH)
loaded_metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
sample = X_test.head(3)
sample_predictions = loaded_model.predict(sample)

assert len(sample_predictions) == len(sample)
assert set(sample_predictions).issubset(set(loaded_metadata["classes"]))
print("Artifact reload test passed.")
display(pd.DataFrame({"actual": y_test.loc[sample.index], "predicted": sample_predictions}, index=sample.index))


## 13. App readiness and deployment handoff

Deployment is deliberately a separate checkpoint. Before building `app.py`, confirm:

- The notebook runs top to bottom in a clean project environment.
- Metrics and confusion matrix have been reviewed and interpreted.
- Model and metadata artifacts exist and reload correctly.
- The app input schema matches `metadata["input_features"]` and training dtypes.
- Tests cover default inputs, unknown categories, missing values, and missing/corrupt artifacts.
- Dataset download and dependency installation work from the documented instructions.
- Secrets and private data are not committed.

Then we will create the polished Streamlit UI, test it locally, push code and required artifacts, configure Streamlit Community Cloud's repository/branch/main-file path, and verify the public URL. Do not claim the app is deployed until its public URL has been opened and tested.

## 14. Generate downloadable HTML and PDF

After executing and reviewing every cell, save this notebook with outputs. From the repository root, run:

```bash
python tools/export_notebook.py "02-decision-tree/01-telecom-customer-churn/notebooks/decision_tree_churn.ipynb"
```

The exporter executes a temporary copy and writes HTML/PDF under the notebook's `docs/` folder. Review page breaks, charts and code colours before committing the exports. If a cell fails, fix the error instead of publishing a notebook with misleading or missing outputs.
